# Banking77 LoRA Fine-Tuning — Colab Orchestration
Clones the repo, installs deps (leaving Colab's preinstalled torch alone per outside-voice E4), and runs the pipeline: data prep -> baseline eval -> LoRA training (4 configs) -> validation-based selection -> one fine-tuned test eval -> report generation.

Checkpoints and eval predictions both live on Google Drive, so a disconnect loses at most one epoch of training or one batch of eval. Every cell is safe to re-run: finished work is skipped, partial work resumes.

In [ ]:
# Always start from a fresh clone of main (a stale `repo/` dir makes `git clone` fail and silently keeps old code).
%cd /content
!rm -rf repo
!git clone https://github.com/V1cky16112003/banking77-lora-finetuning.git repo
%cd repo
!git log -1 --oneline

In [ ]:
# Install deps EXCEPT torch (Colab's preinstalled CUDA-matched torch build stays as-is, per E4).
!pip install -r <(grep -v '^torch' requirements.txt)

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
CHECKPOINT_ROOT = Path('/content/drive/MyDrive/banking77-lora-checkpoints')
# Predictions on Drive too, so evaluate_dataset's incremental JSONL resume survives a runtime reset.
EVAL_ROOT = Path('/content/drive/MyDrive/banking77-lora-eval')
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

## Step 1: Load config and data (design-doc step 1-2)

In [ ]:
from src.config import load_task_config
from src.data import load_banking77_splits

config = load_task_config()
# Raises if configs/task.yaml's 77 labels don't exactly match the loaded dataset.
splits = load_banking77_splits(config)

print(f'{len(splits.labels)} labels, dataset revision: {splits.dataset_revision}')
print(f'train={len(splits.train)} val={len(splits.val)} test={len(splits.test)} (frozen, untouched until final eval)')

## Step 2: Baseline evaluation (zero-shot) on the frozen test set (design-doc step 3)

In [ ]:
import gc
import torch
from transformers import AutoTokenizer
from src.evaluate import evaluate_dataset
from src.evaluate_core import compute_macro_f1
from src.train import MODEL_NAME, load_base_model

def to_examples(ds):
    return [{'text': ex['text'], 'label': splits.labels[ex['label']]} for ex in ds]

def macro_f1(results):
    return compute_macro_f1([t for t, _ in results], [p for _, p in results], config.labels)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
test_examples = to_examples(splits.test)

base_model = load_base_model(use_qlora=False)
base_model.eval()
baseline_results = evaluate_dataset(
    base_model, tokenizer, test_examples, config, EVAL_ROOT / 'baseline_test_predictions.jsonl'
)
print('baseline test macro-F1:', round(macro_f1(baseline_results), 4))

# Free the GPU before training loads its own copy of the model.
del base_model
gc.collect()
torch.cuda.empty_cache()

## Step 3: LoRA fine-tuning -- 4 candidate configs (3 bf16 LoRA + 1 QLoRA) (design-doc step 4)
Safe to re-run after a disconnect: finished configs are skipped, a partial one resumes from its last epoch checkpoint.

In [ ]:
from src.train import CANDIDATE_CONFIGS, train_lora

for run_config in CANDIDATE_CONFIGS:
    train_lora(config, splits, run_config, CHECKPOINT_ROOT / run_config.name)
    gc.collect()
    torch.cuda.empty_cache()

## Step 4: Evaluate each config on validation, pick the best, confirm once on the frozen test set (design-doc step 5)

In [ ]:
import json
from peft import PeftModel

def load_finetuned(run_config):
    # QLoRA adapters are evaluated on the same 4-bit base they were trained on.
    base = load_base_model(run_config.use_qlora)
    model = PeftModel.from_pretrained(base, str(CHECKPOINT_ROOT / run_config.name))
    model.eval()
    return model

val_examples = to_examples(splits.val)
val_scores = {}
for run_config in CANDIDATE_CONFIGS:
    model = load_finetuned(run_config)
    results = evaluate_dataset(model, tokenizer, val_examples, config, EVAL_ROOT / f'val_{run_config.name}.jsonl')
    val_scores[run_config.name] = macro_f1(results)
    print(f'{run_config.name}: val macro-F1 = {val_scores[run_config.name]:.4f}')
    del model
    gc.collect()
    torch.cuda.empty_cache()

best = max(CANDIDATE_CONFIGS, key=lambda rc: val_scores[rc.name])
print('Selected on validation:', best.name)
(EVAL_ROOT / 'val_scores.json').write_text(json.dumps({'scores': val_scores, 'selected': best.name}, indent=2))

# The frozen test set is touched exactly once, by the validation-selected config only.
model = load_finetuned(best)
finetuned_results = evaluate_dataset(
    model, tokenizer, test_examples, config, EVAL_ROOT / 'finetuned_test_predictions.jsonl'
)
print(f'{best.name}: test macro-F1 = {macro_f1(finetuned_results):.4f}')

## Step 5: Generate the report (results table, per-class F1, confusion heatmap, bootstrap CI) -- item 2/3, E1

In [ ]:
# `-m` from the repo root so `import src` resolves.
!python -m scripts.generate_report {EVAL_ROOT}/baseline_test_predictions.jsonl --output-dir {EVAL_ROOT}/report_baseline
!python -m scripts.generate_report {EVAL_ROOT}/finetuned_test_predictions.jsonl --output-dir {EVAL_ROOT}/report_finetuned

## Step 6: Download the adapter checkpoint locally for serving (D5)
Download `banking77-lora-checkpoints/<selected config>/` from Drive, then run `ADAPTER_DIR=<that dir> uvicorn src.serve:app` locally.